# Objective

Built a chatbot to call tools using yfinance library, and incorporate mcp server and client.

Here we also extend the MCP chatbot capabilities by making it connect to other MCP servers:
1. Fetch server
2. Filesystem server

## Tool Functions

In [7]:
%%writefile stock_mcp_server.py
import yfinance as yf

from mcp.server.fastmcp import FastMCP

# Initialize FastMCP server
mcp = FastMCP("stock_mcp")

@mcp.tool()
def get_price(stock_code: str) -> float:
    """
    Get stock data from Yahoo Finance.

    Args:
        stock_code (str): The stock code to fetch data for.

    Returns:
        float: The current stock price.
    """

    stk = yf.Ticker(stock_code)

    try:
        stk_price = stk.info['currentPrice']
    except:
        try:
            stk_price = stk.info['previousClose']
        except:
            print("Error!!!!!! Unable to get stk_price.")

    return stk_price

@mcp.tool()
def get_analyst_price_target(stock_code: str) -> float:
    """
    Get analyst price targets from Yahoo Finance.

    Args:
        stock_code (str): The stock code to fetch data for.

    Returns:
        float: The mean analyst price target.
    """
    stk = yf.Ticker(stock_code)

    try:
        price_target = stk.analyst_price_targets['mean']
    except:
        print("Error!!!!!! Unable to get analyst price targets.")

    return price_target

if __name__ == "__main__":
    mcp.run(transport='stdio')

Overwriting stock_mcp_server.py


## Chatbot Code

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

In [ ]:
%%writefile stock_mcp_client_ref_servers.py

from dotenv import load_dotenv
from anthropic import Anthropic
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from typing import List, Dict, TypedDict
from contextlib import AsyncExitStack
import json
import asyncio

load_dotenv()

class ToolDefinition(TypedDict):
    name: str
    description: str
    input_schema: dict

class MCP_ChatBot:

    def __init__(self):
        # Initialize session and client objects
        self.sessions: List[ClientSession] = [] # new
        self.exit_stack = AsyncExitStack() # new
        self.anthropic = Anthropic()
        self.available_tools: List[ToolDefinition] = [] # new
        self.tool_to_session: Dict[str, ClientSession] = {} # new


    async def connect_to_server(self, server_name: str, server_config: dict) -> None:
        """Connect to a single MCP server."""
        try:
            server_params = StdioServerParameters(**server_config)
            stdio_transport = await self.exit_stack.enter_async_context(
                stdio_client(server_params)
            ) # new
            read, write = stdio_transport
            session = await self.exit_stack.enter_async_context(
                ClientSession(read, write)
            ) # new
            await session.initialize()
            self.sessions.append(session)
            
            # List available tools for this session
            response = await session.list_tools()
            tools = response.tools
            print(f"\nConnected to {server_name} with tools:", [t.name for t in tools])
            
            for tool in tools: # new
                self.tool_to_session[tool.name] = session
                self.available_tools.append({
                    "name": tool.name,
                    "description": tool.description,
                    "input_schema": tool.inputSchema
                })
        except Exception as e:
            print(f"Failed to connect to {server_name}: {e}")

    async def connect_to_servers(self): # new
        """Connect to all configured MCP servers."""
        try:
            with open("server_config.json", "r") as file:
                data = json.load(file)
            
            servers = data.get("mcpServers", {})
            
            for server_name, server_config in servers.items():
                await self.connect_to_server(server_name, server_config)
        except Exception as e:
            print(f"Error loading server configuration: {e}")
            raise
    

    async def process_query(self, query):
    
        messages = [{'role': 'user', 'content': query}]
        
        response = self.anthropic.messages.create(max_tokens=1024,
                                                  model="claude-sonnet-4-20250514", 
                                                  tools=self.available_tools, # tools exposed to the LLM
                                                  messages=messages)
        
        process_query = True
        while process_query:
            # Collect ALL content from assistant's response first
            assistant_content = []
            tool_results = []
            
            for content in response.content:
                if content.type == 'text':
                    print(content.text)
                    assistant_content.append(content)
                    
                elif content.type == 'tool_use':
                    assistant_content.append(content)
                    
                    tool_id = content.id
                    tool_args = content.input
                    tool_name = content.name
                    print(f"Calling tool {tool_name} with args {tool_args}")
                    
                    # tool invocation through the client session
                    session = self.tool_to_session[tool_name] # new
                    # print('session for tool ', tool_name, ' = ', session)
                    result = await session.call_tool(tool_name, arguments=tool_args)
                    # print('result of call_tool = ', result)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": tool_id,
                        "content": result.content
                    })
                    
            
            # Now append the assistant message ONCE with all content
            messages.append({'role': 'assistant', 'content': assistant_content})
            
            # If there were tool uses, add tool results and continue
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
                # print('messages after appending tool results = ', messages)
                print('len(messages) after appending tool results = ', len(messages))
                response = self.anthropic.messages.create(max_tokens=1024,
                                                          model="claude-sonnet-4-20250514", 
                                                          tools=self.available_tools,
                                                          messages=messages)
                                                                 
                # Check if we're done
                if len(response.content) == 1 and response.content[0].type == "text":
                    print(response.content[0].text)
                    process_query = False
            else:
                # No tool uses, we're done
                process_query = False 
    
    
    async def chat_loop(self):
        """Run an interactive chat loop"""
        print("\nMCP Chatbot Started!")
        print("Type your queries or 'quit' to exit.")
        
        while True:
            try:
                query = input("\nQuery: ").strip()
        
                if query.lower() == 'quit':
                    break
                    
                await self.process_query(query)
                print("\n")
                    
            except Exception as e:
                print(f"\nError: {str(e)}")
    
    async def cleanup(self): # new
        """Cleanly close all resources using AsyncExitStack."""
        await self.exit_stack.aclose()


async def main():
    chatbot = MCP_ChatBot()
    try:
        # the mcp clients and sessions are not initialized using "with"
        # like in the previous lesson
        # so the cleanup should be manually handled
        await chatbot.connect_to_servers() # new! 
        await chatbot.chat_loop()
    finally:
        await chatbot.cleanup() #new! 


if __name__ == "__main__":
    asyncio.run(main())

Overwriting stock_mcp_client_ref_servers.py


In [ ]:
# Try with query:
# Fetch https://www.google.com/search?q=Apple+stock+analysis+2025+buy+reasons, do a summary, and write the summary to a file 'AAPL.txt'
# Note that many of the websites blocks bots, so the results may vary.

# Resources

- [Quick Start for Client Developpers](https://modelcontextprotocol.io/quickstart/client)
- [Writing MCP client](https://github.com/modelcontextprotocol/python-sdk/blob/main/examples/clients/simple-chatbot/mcp_simple_chatbot/main.py)
- [Another mcp chatbot example](https://github.com/modelcontextprotocol/python-sdk/blob/main/examples/clients/simple-chatbot/mcp_simple_chatbot/main.py)